[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/01_memory_roofline/01_memory_roofline.ipynb)

# 01 · 显存解剖与 Roofline

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实 **Pythia 全家桶**上算训练显存的四块大头，看优化器状态如何吃掉一半，并用 roofline 判断瓶颈。

**你将完成：**
1. 训练显存四块账：params / grads / optimizer / activations
2. 验证 "Adam 训练 ≈ 16 bytes/param"
3. 判断 "pythia-6.9b 全量微调放得进 24GB 4090 吗"
4. 算术强度与 roofline：为什么自回归解码 memory-bound

> 数据：EleutherAI Pythia 160M→12B 的真实 config。

## 0 · config 管线（同模块 00）

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS = {
  "gpt2":"https://huggingface.co/openai-community/gpt2/resolve/main/config.json",
  "pythia-160m":"https://huggingface.co/EleutherAI/pythia-160m/resolve/main/config.json",
  "pythia-410m":"https://huggingface.co/EleutherAI/pythia-410m/resolve/main/config.json",
  "pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
  "pythia-2.8b":"https://huggingface.co/EleutherAI/pythia-2.8b/resolve/main/config.json",
  "pythia-6.9b":"https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
  "pythia-12b":"https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json",
  "gpt-neox-20b":"https://huggingface.co/EleutherAI/gpt-neox-20b/resolve/main/config.json",
}
def load_config(model):
    path=os.path.join(CACHE, f"{model}.json")
    if not os.path.exists(path): urllib.request.urlretrieve(MODELS[model], path)
    c=json.load(open(path)); g=lambda *ks: next(c[k] for k in ks if k in c)
    h=g("hidden_size","n_embd")
    return dict(model=model, L=g("num_hidden_layers","n_layer"), h=h,
        heads=g("num_attention_heads","n_head"), V=g("vocab_size"),
        I=c.get("intermediate_size", 4*h))
def param_count(cfg):
    L,h,V,I=cfg["L"],cfg["h"],cfg["V"],cfg["I"]
    return 2*V*h + L*(4*h*h+4*h + 2*h*I+(I+h) + 4*h) + h
GB=1024**3
print('已加载', len(MODELS), '个真实模型 config 链接')

## 1 · 训练显存四块账（真实 Pythia）

混合精度 Adam：fp16 权重(2) + fp16 梯度(2) + fp32 主权重(4) + fp32 m(4) + fp32 v(4) = 16 字节/参数。

In [ ]:
def training_memory(cfg, B=1, S=2048, bytes_per_act=2):
    P = param_count(cfg)
    weights = P*2; grads = P*2; optim = P*(4+4+4)   # fp32 master+m+v
    # activation 粗略账：c*L*B*S*h + 注意力 B*heads*S^2
    L,h,heads = cfg["L"],cfg["h"],cfg["heads"]
    act = (12*L*B*S*h + 2*L*B*heads*S*S) * bytes_per_act
    return dict(P=P, weights=weights, grads=grads, optim=optim, act=act,
                total=weights+grads+optim+act)

print(f"{'model':12s} {'参数':>7s} {'权重':>7s} {'梯度':>7s} {'优化器':>8s} {'激活':>7s} {'总计':>8s}")
for m in ["pythia-160m","pythia-1.4b","pythia-6.9b","pythia-12b"]:
    r=training_memory(load_config(m))
    print(f"{m:12s} {r['P']/1e9:6.2f}B {r['weights']/GB:6.1f}G {r['grads']/GB:6.1f}G "
          f"{r['optim']/GB:7.1f}G {r['act']/GB:6.1f}G {r['total']/GB:7.1f}G")

## 2 · 验证 16 bytes/param

不含 activation，模型+优化器状态应恰好 = 16·P 字节。

In [ ]:
for m in ["pythia-1.4b","pythia-12b"]:
    cfg=load_config(m); P=param_count(cfg)
    model_optim = P*(2+2+4+4+4)
    print(f"{m:12s}  {model_optim/P:.1f} bytes/param  (= 16 ✓)  共 {model_optim/GB:.1f} GB")
print("\n=> 优化器状态(m,v,fp32主权重) 占了 12/16 = 75%，远超权重本身")

## 3 · 放得进 24GB RTX 4090 吗？

判断 pythia-6.9b 全量微调 vs 推理 在一张 24GB 卡上的可行性。

In [ ]:
CARD = 24*GB
cfg = load_config("pythia-6.9b"); P=param_count(cfg)
ft = training_memory(cfg, B=1, S=2048)["total"]
infer = P*2 + training_memory(cfg,B=1,S=2048)["act"]*0.1   # 推理:权重+少量
print(f"pythia-6.9b 全量微调需 ≈ {ft/GB:.0f} GB  -> 24GB 卡: {'放得下' if ft<CARD else '放不下 ✗'}")
print(f"pythia-6.9b fp16 推理需 ≈ {P*2/GB:.0f} GB(权重) -> 24GB 卡: {'放得下 ✓' if P*2<CARD else '放不下'}")
print("\n=> 消费级 4090 只能做推理或 LoRA，全量训练得上数据中心卡。这就是账本的实战价值。")

## 4 · 算术强度与 roofline

比较大矩阵乘 vs 自回归解码（矩阵×向量）的算术强度，看谁 memory-bound。

In [ ]:
def intensity_matmul(m,n,k):
    flops = 2*m*n*k
    bytes_ = (m*k + k*n + m*n)*2   # fp16 读写
    return flops/bytes_

# 训练时大 matmul (m=n=k=4096) vs 解码时 batch=1 的 matmul (m=1)
big = intensity_matmul(4096,4096,4096)
decode = intensity_matmul(1, 4096, 4096)
# A100: ~312 TFLOP/s fp16, ~2 TB/s -> 拐点强度
ridge = 312e12 / 2e12
print(f"大矩阵乘 算术强度 = {big:.0f} FLOP/byte  vs A100 拐点 {ridge:.0f}  -> {'compute-bound' if big>ridge else 'memory-bound'}")
print(f"解码(m=1) 算术强度 = {decode:.1f} FLOP/byte  -> {'compute-bound' if decode>ridge else 'memory-bound ✗ 算力空转'}")
print("\n=> 自回归逐 token 解码严重 memory-bound，这是 LLM 推理慢的根本原因（模块06 解决）")

## 5 · 显存账逐项拆解（哪一块最大）

把模块讲解里的四块账落到真实 pythia-6.9b 上，逐项算 GB 与占比，验证 **优化器状态是最大的单一块**、模型+优化器恰好 16 bytes/param。这正是 ZeRO（模块 03）优先分片优化器状态的依据。

In [ ]:
# 把 pythia-6.9b 训练显存拆成四块，看谁最大
cfg=load_config("pythia-6.9b"); P=param_count(cfg)
r=training_memory(cfg, B=1, S=2048)
buckets={"权重(fp16,2P)":r["weights"], "梯度(fp16,2P)":r["grads"],
         "优化器(fp32 主权重+m+v,12P)":r["optim"], "激活(B=1,S=2048)":r["act"]}
print(f"pythia-6.9b ({P/1e9:.1f}B) 训练显存四块账:")
for k,v in buckets.items():
    print(f"  {k:30s} {v/GB:6.1f} GB  ({v/r['total']:5.1%})")
print(f"  {'总计':30s} {r['total']/GB:6.1f} GB")
# 自检：优化器状态是最大的单块；模型+优化器(不含act) 恰好 16P
biggest=max(buckets, key=buckets.get)
assert biggest.startswith("优化器"), "优化器状态应是最大的单一块"
assert abs((r["weights"]+r["grads"]+r["optim"])/P - 16) < 1e-6, "模型+优化器应为 16 bytes/param"
print(f"\n=> 最大单块是 {biggest}；这解释了为什么 ZeRO 优先分片优化器状态(模块03)")

---
## ✏️ 练习区

### ✏️ 练习 1：训练显存账

实现 `adam_memory_bytes(P)`：返回混合精度 Adam 训练下、模型+优化器状态的字节数（不含 activation）。

In [ ]:
def adam_memory_bytes(P):
    # TODO: fp16权重2 + fp16梯度2 + fp32主权重4 + fp32 m 4 + fp32 v 4
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测（真实 pythia-1.4b）——
P = param_count(load_config("pythia-1.4b"))
mem = adam_memory_bytes(P)
assert abs(mem/P - 16) < 1e-6, "应为 16 bytes/param"
assert abs(mem/GB - 16*P/GB) < 1e-6
print(f"练习 1 通过 ✓  pythia-1.4b 训练状态 = {mem/GB:.1f} GB")


### ✏️ 练习 2：activation 随 seq 暴涨

实现 `activation_bytes(cfg, B, S)`：返回 activation 字节（含 `12·L·B·S·h` 主项 + `2·L·B·heads·S²` 注意力项，fp16）。
验证：S 翻倍，注意力项变 4 倍。

In [ ]:
def activation_bytes(cfg, B, S):
    # TODO: (12*L*B*S*h + 2*L*B*heads*S*S) * 2 字节
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
cfg=load_config("pythia-1.4b")
a1=activation_bytes(cfg,1,1024); a2=activation_bytes(cfg,1,2048)
assert a2 > 2*a1, "S 翻倍后因 S² 项，activation 增长超过 2 倍"
# batch 线性
assert abs(activation_bytes(cfg,4,1024) - 4*a1) < 1e-3
print(f"练习 2 通过 ✓  S=1024->2048 activation {a1/GB:.2f}G -> {a2/GB:.2f}G (>2x)")


### ✏️ 练习 3：放得进某张卡吗

实现 `fits_for_training(cfg, card_gb, B, S, margin)`：返回 bool。
`margin` 是预留比例（如 0.15）。总需求 = adam状态 + activation，再 ×(1+margin)。

In [ ]:
def fits_for_training(cfg, card_gb, B=1, S=2048, margin=0.15):
    # TODO: (adam_memory_bytes + activation_bytes)*(1+margin) <= card_gb*GB ?
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
assert fits_for_training(load_config("pythia-160m"), 24), "160m 全量训练能进 4090"
assert not fits_for_training(load_config("pythia-6.9b"), 24), "6.9b 全量训练进不了 4090"
# 6.9b 全量训练(16P≈110G + activation22G + 15%余量 ≈143G)单张 80GB A100 也放不下
assert not fits_for_training(load_config("pythia-6.9b"), 80), "6.9b 全量训练放不进单张 80GB A100"
# 但 1.4b 全量训练能进 80GB；6.9b 需要更大预算(多卡/ZeRO，见模块03)
assert fits_for_training(load_config("pythia-1.4b"), 80), "1.4b 全量训练能进 80GB A100"
print("练习 3 通过 ✓  4090 放得下 160m 全量训练；6.9b 单张 80GB 也放不下(需多卡 ZeRO)")


### ✏️ 练习 4：roofline 判断

实现 `roofline_bound(flops, bytes_moved, peak_flops, bw)`：返回 `"compute"` 或 `"memory"`，
依据实际算术强度与拐点强度比较。

In [ ]:
def roofline_bound(flops, bytes_moved, peak_flops, bw):
    # TODO: 强度 = flops/bytes_moved；拐点 = peak_flops/bw；强度>拐点 -> compute
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测（A100: 312 TFLOP/s, 2 TB/s）——
peak, bw = 312e12, 2e12
# 大 matmul 4096³
assert roofline_bound(2*4096**3, (3*4096*4096)*2, peak, bw) == "compute"
# 解码 m=1
assert roofline_bound(2*1*4096*4096, (4096*4096+4096+4096)*2, peak, bw) == "memory"
print("练习 4 通过 ✓  大matmul compute-bound, 解码 memory-bound")


---
## 📖 参考答案

In [ ]:
# 练习 1
def adam_memory_bytes(P):
    return P*(2+2+4+4+4)
print("练习 1 ✓")

In [ ]:
# 练习 2
def activation_bytes(cfg, B, S):
    L,h,heads=cfg["L"],cfg["h"],cfg["heads"]
    return (12*L*B*S*h + 2*L*B*heads*S*S) * 2
print("练习 2 ✓")

In [ ]:
# 练习 3
def fits_for_training(cfg, card_gb, B=1, S=2048, margin=0.15):
    need = (adam_memory_bytes(param_count(cfg)) + activation_bytes(cfg,B,S))*(1+margin)
    return need <= card_gb*GB
print("练习 3 ✓")

In [ ]:
# 练习 4
def roofline_bound(flops, bytes_moved, peak_flops, bw):
    return "compute" if flops/bytes_moved > peak_flops/bw else "memory"
print("练习 4 ✓ —— 这套账本是 systems 面试的硬通货")